# 🧠 MCLDNN — Fading Kanal Koşullarında Eğitim

**Model:** MCLDNN (Multi-Channel LSTM Deep Neural Network)

Bu notebook, MCLDNN modelini 3 farklı fading kanalında eğitir ve AWGN baseline ile karşılaştırır.

| # | Kanal | Dataset Dosyası | Açıklama |
|---|-------|----------------|----------|
| 1 | Rayleigh | `RML2016.10a_rayleigh.pkl` | NLOS ortam |
| 2 | Rician K=3 | `RML2016.10a_rician_K3.pkl` | Orta LOS |
| 3 | Rician K=10 | `RML2016.10a_rician_K10.pkl` | Güçlü LOS |

## 1. Ortam Kurulumu

In [ ]:
import tensorflow as tf
print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

import numpy as np
import matplotlib.pyplot as plt
import pickle, os, time, shutil
print('\n✅ Ortam hazır.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
REPO_URL = 'https://github.com/erigami-sl/AMR-UnderDifferentNoises-DL.git'
PROJECT_DIR = '/content/AMR-UnderDifferentNoises-DL'

if not os.path.exists(PROJECT_DIR):
    !git clone -b dev {REPO_URL}
else:
    !cd {PROJECT_DIR} && git pull origin dev

os.chdir(PROJECT_DIR)
import sys
sys.path.insert(0, PROJECT_DIR)
print(f'✅ Çalışma dizini: {os.getcwd()}')

## 2. Eğitim Konfigürasyonu

In [ ]:
DRIVE_DIR = '/content/drive/MyDrive/AMR-Project'

CHANNEL_CONFIGS = [
    {
        'name': 'rayleigh',
        'display': 'Rayleigh (NLOS)',
        'dataset_file': os.path.join(DRIVE_DIR, 'RML2016.10a_rayleigh.pkl'),
    },
    {
        'name': 'rician_K3',
        'display': 'Rician K=3 (Orta LOS)',
        'dataset_file': os.path.join(DRIVE_DIR, 'RML2016.10a_rician_K3.pkl'),
    },
    {
        'name': 'rician_K10',
        'display': 'Rician K=10 (Güçlü LOS)',
        'dataset_file': os.path.join(DRIVE_DIR, 'RML2016.10a_rician_K10.pkl'),
    },
]

MODEL_NAME = 'mcldnn'
EPOCHS = 1000
BATCH_SIZE = 400
LR = 0.001

for cfg in CHANNEL_CONFIGS:
    exists = os.path.exists(cfg['dataset_file'])
    status = '✅' if exists else '❌'
    print(f"{status} {cfg['display']}: {cfg['dataset_file']}")
    if not exists:
        raise FileNotFoundError(f"Dataset bulunamadı: {cfg['dataset_file']}\n"
                                f"Önce 04_generate_faded_datasets.ipynb çalıştırın.")

## 3. Eğitim Döngüsü

Her kanal koşulu için sırayla:
1. Dataset yükle → split → MCLDNN formatına dönüştür
2. Model oluştur (sıfırdan)
3. Eğit (early stopping ile)
4. Değerlendir
5. Sonuçları kaydet

In [ ]:
from src.utils.dataset import load_data, prepare_data_mcldnn
from src.models.mcldnn import MCLDNN
from src.utils.metrics import evaluate_model, plot_training_history

all_results = {}

for cfg in CHANNEL_CONFIGS:
    ch_name = cfg['name']
    ch_display = cfg['display']
    
    print('\n' + '=' * 70)
    print(f'🔄 EĞİTİM BAŞLIYOR: {ch_display}')
    print('=' * 70)
    
    # --- 1. Dataset Yükleme ---
    print(f'\n📂 Dataset yükleniyor: {cfg["dataset_file"]}')
    (mods, snrs, lbl), (X_train, Y_train), (X_val, Y_val), \
        (X_test, Y_test), (train_idx, val_idx, test_idx) = load_data(cfg['dataset_file'])
    
    data = prepare_data_mcldnn(X_train, X_val, X_test)
    print(f'   Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')
    
    # --- 2. Model Oluşturma (sıfırdan) ---
    tf.keras.backend.clear_session()
    model = MCLDNN(weights=None, classes=len(mods))
    model.compile(
        loss='categorical_crossentropy',
        metrics=['accuracy'],
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR)
    )
    
    # --- 3. Sonuç dizinleri ---
    RESULTS_DIR = os.path.join(PROJECT_DIR, 'results', MODEL_NAME, ch_name)
    os.makedirs(os.path.join(RESULTS_DIR, 'weights'), exist_ok=True)
    os.makedirs(os.path.join(RESULTS_DIR, 'figures'), exist_ok=True)
    os.makedirs(os.path.join(RESULTS_DIR, 'predictions'), exist_ok=True)
    WEIGHTS_PATH = os.path.join(RESULTS_DIR, 'weights', 'best_weights.keras')
    
    # --- 4. Eğitim ---
    callbacks = [
        tf.keras.callbacks.ModelCheckpoint(
            WEIGHTS_PATH, monitor='val_loss',
            verbose=1, save_best_only=True, mode='auto'
        ),
        tf.keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss', factor=0.5,
            verbose=1, patience=5, min_lr=1e-7
        ),
        tf.keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=50,
            verbose=1, mode='auto', restore_best_weights=True
        )
    ]
    
    t0 = time.time()
    history = model.fit(
        data['train'], Y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        verbose=2,
        validation_data=(data['val'], Y_val),
        callbacks=callbacks
    )
    train_time = time.time() - t0
    print(f'\n⏱️ Eğitim süresi: {train_time/60:.1f} dakika')
    
    # --- 5. Eğitim grafikleri ---
    plot_training_history(history, save_dir=os.path.join(RESULTS_DIR, 'figures'))
    
    # --- 6. Değerlendirme ---
    model.load_weights(WEIGHTS_PATH)
    acc, acc_mod_snr = evaluate_model(
        model=model,
        X_test_inputs=data['test'],
        Y_test=Y_test,
        lbl=lbl,
        test_idx=test_idx,
        snrs=snrs,
        classes=mods,
        batch_size=BATCH_SIZE,
        results_dir=RESULTS_DIR
    )
    
    all_results[ch_name] = {'acc': acc, 'acc_mod_snr': acc_mod_snr, 'time': train_time}
    
    # --- 7. Drive'a kaydet ---
    DRIVE_RESULTS = os.path.join(DRIVE_DIR, 'results', f'{MODEL_NAME}_{ch_name}')
    os.makedirs(DRIVE_RESULTS, exist_ok=True)
    shutil.copy2(WEIGHTS_PATH, os.path.join(DRIVE_RESULTS, 'best_weights.keras'))
    with open(os.path.join(DRIVE_RESULTS, 'acc.pkl'), 'wb') as f:
        pickle.dump(acc, f)
    with open(os.path.join(DRIVE_RESULTS, 'acc_mod_snr.pkl'), 'wb') as f:
        pickle.dump(acc_mod_snr, f)
    print(f'💾 Sonuçlar kaydedildi: {DRIVE_RESULTS}')

print('\n' + '=' * 70)
print('🎉 TÜM EĞİTİMLER TAMAMLANDI!')
print('=' * 70)

## 4. Karşılaştırmalı Sonuçlar

In [ ]:
# AWGN baseline sonuçlarını yükle (varsa)
awgn_acc_path = os.path.join(DRIVE_DIR, 'results', f'{MODEL_NAME}_awgn', 'acc.pkl')
if os.path.exists(awgn_acc_path):
    with open(awgn_acc_path, 'rb') as f:
        all_results['awgn'] = {'acc': pickle.load(f)}
    print('✅ AWGN baseline sonuçları yüklendi.')
else:
    print('⚠️ AWGN baseline sonuçları bulunamadı, karşılaştırmada yer almayacak.')

In [ ]:
# SNR vs Accuracy karşılaştırma grafiği
colors = {'awgn': '#4CAF50', 'rayleigh': '#F44336', 'rician_K3': '#FF9800', 'rician_K10': '#2196F3'}
labels = {'awgn': 'AWGN (Baseline)', 'rayleigh': 'Rayleigh', 'rician_K3': 'Rician K=3', 'rician_K10': 'Rician K=10'}

plt.figure(figsize=(12, 7))
for ch_name, result in all_results.items():
    acc = result['acc']
    plt.plot(snrs, [acc[s] for s in snrs], 'o-',
             label=labels.get(ch_name, ch_name),
             color=colors.get(ch_name, '#999'),
             linewidth=2, markersize=5)

plt.xlabel('SNR (dB)', fontsize=13)
plt.ylabel('Classification Accuracy', fontsize=13)
plt.title(f'MCLDNN — SNR vs Accuracy (Tüm Kanal Koşulları)', fontsize=15)
plt.legend(fontsize=11)
plt.grid(True, alpha=0.3)
plt.ylim([0, 1])
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_DIR, 'results', f'{MODEL_NAME}_comparison.png'), dpi=300)
plt.show()

# Özet tablo
print(f'\n{"Kanal":>20} | {"Ort. Accuracy":>14} | {"En Yüksek":>10} | {"En Düşük":>10}')
print('-' * 65)
for ch_name, result in all_results.items():
    acc_vals = list(result['acc'].values())
    print(f'{labels.get(ch_name, ch_name):>20} | {np.mean(acc_vals)*100:>13.2f}% | {max(acc_vals)*100:>9.2f}% | {min(acc_vals)*100:>9.2f}%')

---
## ✅ Tamamlandı

MCLDNN fading kanal eğitimleri tamamlandı.

**Sonraki adım:** Phase 3 — Tüm sonuçları topla, karşılaştır ve raporla.